# SGD vs Adam on a 2D Loss Landscape

Different optimizers take different paths toward a minimum.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Loss surface f(w1, w2) = w1² + 10·w2²

In [ ]:
def loss_fn(w):
    return w[0]**2 + 10 * w[1]**2

def grad_fn(w):
    return torch.tensor([2*w[0], 20*w[1]], dtype=torch.float32)

w1, w2 = np.meshgrid(np.linspace(-2, 2, 80), np.linspace(-2, 2, 80))
Z = w1**2 + 10 * w2**2

## 2. Contour plot of the bowl

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(w1, w2, Z, levels=20, cmap='viridis')
ax.set_xlabel('w1'); ax.set_ylabel('w2'); ax.set_title('Loss landscape (elongated bowl)')
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 3. Run SGD and Adam from same start

In [ ]:
def optimize(optimizer_cls, steps=40, lr=0.1, **kwargs):
    w = torch.tensor([1.8, 1.5], requires_grad=True)
    opt = optimizer_cls([w], lr=lr, **kwargs)
    path = [w.detach().clone().numpy()]
    for _ in range(steps):
        l = loss_fn(w)
        opt.zero_grad()
        l.backward()
        opt.step()
        path.append(w.detach().clone().numpy())
    return np.array(path)

path_sgd = optimize(torch.optim.SGD, lr=0.08)
path_adam = optimize(torch.optim.Adam, lr=0.3)

## 4. Overlay optimizer paths

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.contour(w1, w2, Z, levels=20, cmap='viridis', alpha=0.7)
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'r.-', label='SGD', lw=2)
ax.plot(path_adam[:, 0], path_adam[:, 1], 'b.-', label='Adam', lw=2)
ax.scatter([0], [0], c='gold', s=100, zorder=5, edgecolors='k', label='min')
ax.legend(); ax.set_aspect('equal')
ax.set_title('Optimizer trajectories on f(w)=w1²+10w2²')
plt.tight_layout(); plt.show()

## 5. Loss vs step

In [ ]:
def loss_path(path):
    return [loss_fn(torch.tensor(p)).item() for p in path]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_path(path_sgd), 'r-', label='SGD', lw=2)
ax.plot(loss_path(path_adam), 'b-', label='Adam', lw=2)
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend()
ax.set_title('Loss convergence comparison')
plt.tight_layout(); plt.show()